# 04 — Measurement & the Born Rule

Turn amplitudes into the probabilities a measurement actually returns — and discover the one thing a measurement cannot see.

**Prerequisites:** `01/01_amplitudes_and_superposition` (amplitudes, weights $|a|^2$), `01/02_why_amplitudes_are_complex` (relative phase, the recombination rule), `01/03_the_bloch_sphere` (latitude $z$, longitude $\phi$).

**What you'll be able to do**
- State the Born rule — an amplitude becomes a probability through $|a_x|^2$ — and compute it by hand and with `born_probabilities`.
- Read a measurement as a *shadow*: it turns a qubit into a probability vector (the object of `00/02`) and drops the rest.
- Explain why a computational-basis ($Z$) measurement is blind to a state's phase, and how a different measurement can see it.

You can write a qubit's amplitudes, but you can never read them off directly. Nature only lets you *measure*, and a measurement returns a plain $0$ or $1$, at random. The rule connecting the two is short, exact, and one of the most important sentences in the course. It is also the promised link between the amplitudes of `01/01` and the probability weights you transported in `00/02`.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from qot_course import viz
from qot_course.colors import COLORS
from qot_course.quantum.states import KET_0, KET_1, KET_PLUS, KET_MINUS, qubit_state, born_probabilities, bloch_vector

viz.use_course_style()

## 1. The Born rule

When you measure a qubit in the computational basis (the $|0\rangle,|1\rangle$ basis — a **$Z$ measurement**, after the axis it reads on the Bloch sphere), you get outcome $x \in \{0, 1\}$ with probability equal to the **squared size of its amplitude**:

$$ P(x) = |a_x|^2 . $$

This is the **Born rule**. The "weights" you have been computing since `01/01` were this rule all along; today they get their physical meaning. Two consequences are worth saying out loud. First, measurement is **random**: even a perfectly known state yields an unpredictable single outcome; only the *probabilities* are fixed by the state. Second, because we take the modulus, the probabilities depend on the amplitude's *size* — and, as we are about to see, not at all on its phase.

(One more thing a measurement does — it *changes* the state, leaving it in $|x\rangle$ after reporting $x$. That is why every shot in `01/11` needs a freshly prepared copy: you cannot measure the same qubit twice to learn more. Today we only need the probabilities.)

Compute the rule by hand for three states, then with the course helper `born_probabilities`, and check they agree.

In [ ]:
psi = qubit_state(theta=2 * np.pi / 3, phi=np.pi / 4)   # the tilted state of 01/01-01/03
for name, state in [("|0>", KET_0), ("|+>", KET_PLUS), ("|psi>", psi)]:
    a0, a1 = state
    p0_hand, p1_hand = abs(a0) ** 2, abs(a1) ** 2            # Born rule by hand
    p = born_probabilities(state)                            # course helper
    print(f"{name:5s} by hand: P(0) = {p0_hand:.3f}  P(1) = {p1_hand:.3f}   "
          f"helper: P(0) = {p['0']:.3f}  P(1) = {p['1']:.3f}   sum = {p0_hand + p1_hand:.3f}")

**Read the output.** By hand and by helper agree to every digit. $|0\rangle$ always gives $0$ ($P(0)=1$) — a classical state is certain. $|+\rangle$ is a perfect coin flip, $0.5/0.5$ — the equal superposition is maximally uncertain. The tilted $|\psi\rangle$ lands in between, $0.25/0.75$, and every row sums to $1$: normalisation (`01/01`) is exactly what makes the Born rule a probability distribution. Let's see all three as a picture.

In [ ]:
states = {"|0>": KET_0, "|+>": KET_PLUS, "|psi>": psi}
p0 = [born_probabilities(s)["0"] for s in states.values()]
p1 = [born_probabilities(s)["1"] for s in states.values()]

fig, ax = plt.subplots(figsize=(7.5, 4.2))
x = np.arange(len(states))
width = 0.38
ax.bar(x - width / 2, p0, width, color=COLORS["source"], label="P(0)")
ax.bar(x + width / 2, p1, width, color=COLORS["target"], label="P(1)")
ax.set_xticks(x)
ax.set_xticklabels(list(states))
ax.set_ylabel("probability")
ax.set_ylim(0, 1)
ax.set_title("Born-rule probabilities for a Z measurement", pad=12)
ax.legend()
plt.show()

**Read the figure.** $|0\rangle$ (periwinkle bar only) puts all its probability on outcome $0$. $|+\rangle$ splits $50/50$ — a fair coin. $|\psi\rangle$ leans to outcome $1$ (amber). For every state the two bars sum to $1$: you always get *some* outcome. This pair of numbers is the whole story of what a single $Z$ measurement can tell you about a state.

## 2. The measurement as a shadow: from qubit to probability vector

Look at what the Born rule hands you: two non-negative numbers that sum to one. That is a **probability vector** — exactly the object you transported in `00/02`. So a $Z$ measurement is a map

$$ (a_0, a_1)\ \longmapsto\ p = \big(|a_0|^2,\ |a_1|^2\big), $$

from the qubit's two complex amplitudes (four real numbers, of which normalisation removes one and the invisible global phase of `01/02` another) down to one real number, $P(0)$ (since $P(1) = 1 - P(0)$). Call $p$ the **shadow** the measurement casts of the state. On the Bloch sphere of `01/03` this shadow is the *latitude* alone: add and subtract the two identities $|a_0|^2 + |a_1|^2 = 1$ and $|a_0|^2 - |a_1|^2 = z$ to get

$$ P(0) = \frac{1+z}{2}, \qquad P(1) = \frac{1-z}{2} , $$

so the $Z$ measurement reads the height of the point and nothing else. Let's check on the tilted state.

In [ ]:
x_b, y_b, z_b = bloch_vector(psi)
p = born_probabilities(psi)
print(f"Bloch z of |psi>       = {z_b:.4f}")
print(f"(1+z)/2, (1-z)/2       = {(1 + z_b)/2:.4f}, {(1 - z_b)/2:.4f}")
print(f"Born P(0), P(1)        = {p['0']:.4f}, {p['1']:.4f}")
print(f"P(0) - P(1) == z       : {np.isclose(p['0'] - p['1'], z_b)}")
print(f"the shadow p is a probability vector: non-negative {min(p.values()) >= 0}, sums to {p['0'] + p['1']:.4f}")

**Read the output.** $P(0)-P(1)$ is exactly the Bloch $z$: the Born rule and the sphere's latitude are the same number seen twice. The two remaining coordinates $(x, y)$ — the longitude, i.e. the phase — do not enter. That is the crack we now open on purpose.

## 3. What the measurement cannot see

Keep a state on the equator ($\theta = \pi/2$, an equal superposition) and vary only the phase $\phi$. Geometrically — from `01/03` — this **spins the state around the equator**, landing on a genuinely different point of the Bloch sphere each time ($|+\rangle$, $|{+i}\rangle$, $|-\rangle$). Does the $Z$ measurement notice?

In [ ]:
for phi in [0.0, np.pi / 2, np.pi]:
    state = qubit_state(theta=np.pi / 2, phi=phi)   # equator; only the phase changes
    p = born_probabilities(state)
    print(f"phi = {phi:.2f}:  P(0) = {p['0']:.3f}   P(1) = {p['1']:.3f}   Bloch (x, y, z) = {np.round(bloch_vector(state), 3)}")

**Read the output.** The phase $\phi$ sweeps the state a quarter- and then a half-turn around the equator — the Bloch $x$ and $y$ change completely — yet $P(0)$ and $P(1)$ never move from $0.5$. A computational-basis measurement is **blind to phase**. The information is real: `01/02` showed that $|+\rangle$ and $|-\rangle$ behave *oppositely* under the recombination rule, and it lives in the off-diagonal of the density matrix you will build in `01/12`. But a plain $Z$ measurement throws it away.

Can *any* measurement see it? Yes — and you already have the tool. Run the recombination rule of `01/02` first, *then* measure in $Z$: that two-step procedure is a measurement along the $x$-axis of the sphere (an "$X$ measurement", made official in `01/08`; exercise 4 has you prove that this shadow reads the Bloch $x$ exactly as $Z$ reads $z$). Let's compare the two shadows of $|+\rangle$ and $|-\rangle$.

In [ ]:
def recombine(state):
    """The by-hand rule of 01/02: (a0, a1) -> ((a0 + a1)/sqrt2, (a0 - a1)/sqrt2)."""
    a0, a1 = state
    return np.array([a0 + a1, a0 - a1]) / np.sqrt(2)

print("state   Z-shadow (P0, P1)      recombine-then-Z shadow (P0, P1)")
for name, state in [("|+>", KET_PLUS), ("|->", KET_MINUS), ("|0>", KET_0)]:
    pz = born_probabilities(state)
    px = born_probabilities(recombine(state))
    print(f"{name:6s} ({pz['0']:.2f}, {pz['1']:.2f})              ({px['0']:.2f}, {px['1']:.2f})")

**Read the output.** Under the $Z$ shadow, $|+\rangle$ and $|-\rangle$ are identical coins and $|0\rangle$ is certain. Under the recombine-then-$Z$ shadow the roles swap: $|+\rangle$ and $|-\rangle$ become *certain and opposite*, while $|0\rangle$ becomes the coin. No single measurement sees everything — each casts one shadow: $Z$ reads the latitude $z$, recombine-then-$Z$ reads $x$, and the $\pm$ sign of $|+\rangle/|-\rangle$ shows up in the second precisely where it is invisible in the first. Neither sees $y$ (exercise 4 finds a state that is a fair coin under *both*); a third shadow, `01/08`–`01/09`, is needed for that. Remember this: the gap between "the full quantum state" and "one measurement's shadow" is the gap this course keeps returning to. In the capstone (module `04`) we ask which shadow an *embedding* writes into a state — the choice of embedding, not quantum mechanics itself, decides what a quantum transport distance can and cannot see; whether anything specifically quantum is needed there is a question we test, not one we settle today.

## Your turn

1. **Bias on demand.** Find a $\theta$ that makes $P(0) = 0.9$ exactly. (Hint: $P(0) = \cos^2(\theta/2)$ — derive this from `qubit_state`'s amplitudes before you use it.) Check with `born_probabilities`, and read off the Bloch $z$ you expect.
2. **Certainty lives at the poles.** Confirm $|1\rangle$ gives $P(1) = 1$. Which states are perfectly *certain* under a $Z$ measurement, and where do they sit on the Bloch sphere? Which states are perfectly certain under recombine-then-$Z$?
3. **Phase versus latitude.** Take `qubit_state(theta, phi)` for a fixed $\theta$ of your choice and sweep $\phi$ over a grid — confirm with `np.allclose` that $P(0)$ never moves, and that the Bloch $z$ never moves either while $x, y$ do. In one sentence: why is $P(0)$ tied to latitude, not longitude?
4. **Harder: two shadows, one state.** From `01/03`, exercise 4, you know $P(1) = (1 - z)/2$. Show that recombine-then-$Z$ gives $P(0) = (1 + x)/2$ (use the amplitude formula $x = 2\,\mathrm{Re}(\bar a_0 a_1)$). Then find the *pure* states whose two shadows are *both* fair coins (there are two), and say where they live on the sphere — you met them in `01/02`–`01/03`.

## What you built

- You stated and used the Born rule, $P(x) = |a_x|^2$, by hand and with `born_probabilities`, and saw normalisation is what makes it a probability distribution.
- You read a measurement as a **shadow**: it turns a qubit into the probability vector of `00/02`, and that shadow is the Bloch latitude, $P(0) = (1+z)/2$.
- You discovered a $Z$ measurement is blind to phase — and that recombine-then-$Z$ (an $X$ measurement) sees *part* of what $Z$ drops, the $x$ coordinate, while $y$ needs a third measurement (`01/08`–`01/09`): no single shadow, and not even these two, sees everything.

That last point is not a footnote — it is the first crack of daylight between "the full quantum state" and "what one measurement reports". Much of the back half of the course lives in that gap, and you found it today.

**Dictionary so far** (the table from `01/03`, plus the row you earned today):

| Classical | Quantum |
|-----------|---------|
| bit ($0$ or $1$) | qubit $a_0\,|0\rangle + a_1\,|1\rangle$ (two complex amplitudes) |
| probabilities only add (non-negative) | amplitudes add *and cancel* — interference; a relative phase $e^{i\varphi}$ is physical, a global phase is not |
| a bit's state space: two isolated points | a pure qubit's state space: the Bloch **sphere** (poles = the classical bit; equator = equal superpositions) |
| outcome probabilities $p_x$ | **Born rule** $p_x = \lvert a_x\rvert^2$ — the *shadow* one measurement casts of a state; a different measurement casts a different shadow |
| probability vector `p` | density matrix $\rho$ (diagonal in a shared basis $\Rightarrow$ classical) |
| marginal | partial trace |
| transport plan | quantum coupling |

(The last three rows are still forward-looking — earned in `01/12`, `01/16` and `04/04`.)

## What's next

We used the bracket $|\cdot\rangle$ for four notebooks and the phrase "as different as $|0\rangle$ and $|1\rangle$" twice. In `01/05_inner_product_and_dirac` we make both precise: the inner product $\langle\varphi|\psi\rangle$, whose modulus squared is — by the Born rule you just learned — the probability of finding $|\psi\rangle$ in state $|\varphi\rangle$. Later, in `01/11`, we sample the Born rule one shot at a time and meet shot noise.

## References

- M. Born, "Zur Quantenmechanik der Stoßvorgänge", *Zeitschrift für Physik* 37, 863–867 (1926). doi:10.1007/BF01397477
- M. A. Nielsen & I. L. Chuang, *Quantum Computation and Quantum Information*, §2.2.3–2.2.5 (measurement postulate), Cambridge University Press (2010). doi:10.1017/CBO9780511976667

**Previous:** `notebooks/01_QuantumFoundations/03_the_bloch_sphere.ipynb` · **Next:** `notebooks/01_QuantumFoundations/05_inner_product_and_dirac.ipynb`